In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import string
import re

# Load the dataset
file_path = "newalternatedataset.csv"
df = pd.read_csv(file_path)

# Preprocessing function
def preprocess_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces and newlines
    return text

# Apply preprocessing to the 'text' column
df['text'] = df['text'].apply(preprocess_text)

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=500)  # Limit features to 500 (you can adjust this)
tfidf_matrix = tfidf.fit_transform(df['text'])

# Convert the TF-IDF matrix to a DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())

# Combine TF-IDF embeddings with original data
result_df = pd.concat([df, tfidf_df], axis=1)

# Save to a new CSV
output_path = "embedded_newalternatedataset.csv"
result_df.to_csv(output_path, index=False)

print(f"Processed dataset saved to {output_path}")


Processed dataset saved to embedded_newalternatedataset.csv


In [ ]:
#biLSTM
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Bidirectional, Dropout, Reshape
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Load the dataset
data_file = 'embedded_newalternatedataset.csv'  # Path to the new TF-IDF embedded CSV
data = pd.read_csv(data_file)

# Extract TF-IDF embeddings and labels
embeddings = data.iloc[:, -500:].values  # Assuming the last 500 columns are TF-IDF features
labels = data[['provokingviolence', 'individualharrassment', 'emotionaldistress']]  # Adjust for actual label columns

# Convert labels to categorical
Y_provoking = pd.get_dummies(labels['provokingviolence']).values
Y_harassment = pd.get_dummies(labels['individualharrassment']).values
Y_distress = pd.get_dummies(labels['emotionaldistress']).values

# Split data into training and test sets
X_train, X_test, Y_train_provoking, Y_test_provoking = train_test_split(embeddings, Y_provoking, test_size=0.30, random_state=1)
_, _, Y_train_harassment, Y_test_harassment = train_test_split(embeddings, Y_harassment, test_size=0.30, random_state=1)
_, _, Y_train_distress, Y_test_distress = train_test_split(embeddings, Y_distress, test_size=0.30, random_state=1)

# Model architecture
input_layer = Input(shape=(embeddings.shape[1],))  # Adjust to TF-IDF vector size
x = Reshape((1, embeddings.shape[1]))(input_layer)  # Reshape to 3D for LSTM (batch_size, 1, vector_size)
x = Dropout(0.3)(x)
x = Bidirectional(LSTM(128, dropout=0.3, recurrent_dropout=0.3))(x)  # Adjust LSTM units for TF-IDF size

# Define separate output layers
output_provoking = Dense(Y_provoking.shape[1], activation='softmax', name='provokingviolence')(x)
output_harassment = Dense(Y_harassment.shape[1], activation='softmax', name='individualharrassment')(x)
output_distress = Dense(Y_distress.shape[1], activation='softmax', name='emotionaldistress')(x)

# Compile the model
model = Model(inputs=input_layer, outputs=[output_provoking, output_harassment, output_distress])
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics={
                  'provokingviolence': ['accuracy'],
                  'individualharrassment': ['accuracy'],
                  'emotionaldistress': ['accuracy']
              })

print(model.summary())

# Training the model
epochs = 20
batch_size = 64
history = model.fit(
    X_train,
    [Y_train_provoking, Y_train_harassment, Y_train_distress],
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    callbacks=[EarlyStopping(monitor='val_loss', min_delta=0.0001, patience=3)]
)

# Evaluate the model
test_results = model.evaluate(X_test, [Y_test_provoking, Y_test_harassment, Y_test_distress])
print(f"Evaluation Results: {test_results}")

# Predictions and reports
preds_provoking, preds_harassment, preds_distress = model.predict(X_test)

print("Classification Report for Provoking Violence:")
print(classification_report(Y_test_provoking.argmax(axis=1), preds_provoking.argmax(axis=1)))

print("Classification Report for Individual Harassment:")
print(classification_report(Y_test_harassment.argmax(axis=1), preds_harassment.argmax(axis=1)))

print("Classification Report for Emotional Distress:")
print(classification_report(Y_test_distress.argmax(axis=1), preds_distress.argmax(axis=1)))


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1             │ (None, 500)            │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_1 (Reshape)       │ (None, 1, 500)         │              0 │ input_layer_1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_1 (Dropout)       │ (None, 1, 500)         │              0 │ reshape_1[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ bidirectional_1           │ (None, 256)            │        644,096 │ dropout_1[0][0]        │
│ (Bidirectional)           │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ provokingviolence (Dense) │ (None, 4)              │          1,028 │ bidirectional_1[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ individualharrassment     │ (None, 4)              │          1,028 │ bidirectional_1[0][0]  │
│ (Dense)                   │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ emotionaldistress (Dense) │ (None, 3)              │            771 │ bidirectional_1[0][0]  │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 646,923 (2.47 MB)

 Trainable params: 646,923 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - emotionaldistress_accuracy: 0.6990 - emotionaldistress_loss: 0.7114 - individualharrassment_accuracy: 0.4983 - individualharrassment_loss: 1.1042 - loss: 2.9259 - provokingviolence_accuracy: 0.5648 - provokingviolence_loss: 1.1103 - val_emotionaldistress_accuracy: 0.7309 - val_emotionaldistress_loss: 0.5485 - val_individualharrassment_accuracy: 0.5343 - val_individualharrassment_loss: 0.9506 - val_loss: 2.4134 - val_provokingviolence_accuracy: 0.6383 - val_provokingviolence_loss: 0.9031
Epoch 2/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step - emotionaldistress_accuracy: 0.7266 - emotionaldistress_loss: 0.5554 - individualharrassment_accuracy: 0.5290 - individualharrassment_loss: 0.9635 - loss: 2.4674 - provokingviolence_accuracy: 0.6170 - provokingviolence_loss: 0.9485 - val_emotionaldistress_accuracy: 0.7298 - val_emotionaldistress_loss: 0.5432 - val_individualharrassment_accuracy: 0.5385 - val_individualharrassment_loss: 0.9

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/m

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import xgboost as xgb

# Load the dataset
data = pd.read_csv('embedded_newalternatedataset.csv')

# Separate TF-IDF features (columns excluding original data and target columns)
tfidf_features = [col for col in data.columns if col not in ['text', 'provokingviolence', 'individualharrassment', 'emotionaldistress']]
X = data[tfidf_features].values

# Define the target columns and initialize label encoders for each
target_columns = ['provokingviolence', 'individualharrassment', 'emotionaldistress']
label_encoders = {col: LabelEncoder() for col in target_columns}

# Encode the labels for each target column
y_encoded = {}
for col in target_columns:
    y_encoded[col] = label_encoders[col].fit_transform(data[col])

# Split data into training and validation sets for each target column
train_test_splits = {}
for col in target_columns:
    X_train, X_val, y_train, y_val = train_test_split(X, y_encoded[col], test_size=0.2, random_state=42)
    train_test_splits[col] = (X_train, X_val, y_train, y_val)

# Function to train and evaluate XGBoost for each target
def train_evaluate_xgboost(target_column):
    X_train, X_val, y_train, y_val = train_test_splits[target_column]

    # Initialize XGBoost classifier
    model = xgb.XGBClassifier(
        objective='multi:softmax',
        num_class=len(label_encoders[target_column].classes_),  # Number of classes for the target
        use_label_encoder=False,
        random_state=42,
        tree_method='gpu_hist',  # Enable GPU support
        gpu_id=0,  # Use the first GPU if available
        max_depth=6,
        learning_rate=0.1,
        n_estimators=100
    )

    # Train the model
    model.fit(X_train, y_train)

    # Predict on validation data
    y_pred = model.predict(X_val)

    # Convert predictions and true labels back to original labels
    y_pred_labels = label_encoders[target_column].inverse_transform(y_pred)
    y_val_labels = label_encoders[target_column].inverse_transform(y_val)

    # Print classification report and accuracy
    print(f"Classification Report for '{target_column}':")
    print(classification_report(y_val_labels, y_pred_labels))
    accuracy = accuracy_score(y_val_labels, y_pred_labels)
    print(f"Overall Accuracy for '{target_column}': {accuracy:.4f}\n")

    return model

# Train and evaluate XGBoost model for each target column
models = {}
for col in target_columns:
    print(f"Training and evaluating model for target: {col}")
    models[col] = train_evaluate_xgboost(col)


Training and evaluating model for target: provokingviolence


/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [18:07:40] WARNING: /workspace/src/common/error_msg.cc:45: `gpu_id` is deprecated since2.0.0, use `device` instead. E.g. device=cpu/cuda/cuda:0
  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [18:07:40] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [18:07:41] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [18:07:44] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU t

Classification Report for 'provokingviolence':
              precision    recall  f1-score   support

           0       0.58      0.15      0.24      1975
           1       0.00      0.00      0.00       966
           2       0.62      0.91      0.73      5855
           3       0.78      0.66      0.71      2191

    accuracy                           0.64     10987
   macro avg       0.49      0.43      0.42     10987
weighted avg       0.59      0.64      0.58     10987

Overall Accuracy for 'provokingviolence': 0.6418

Training and evaluating model for target: individualharrassment


/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [18:07:45] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [18:07:45] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [18:07:48] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)


Classification Report for 'individualharrassment':
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        81
           1       0.56      0.18      0.27      2386
           2       0.52      0.83      0.64      5430
           3       0.57      0.30      0.39      3090

    accuracy                           0.53     10987
   macro avg       0.41      0.33      0.33     10987
weighted avg       0.54      0.53      0.49     10987

Overall Accuracy for 'individualharrassment': 0.5335

Training and evaluating model for target: emotionaldistress


/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [18:07:49] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [18:07:49] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Classification Report for 'emotionaldistress':
              precision    recall  f1-score   support

           0       1.00      0.01      0.02       100
           1       0.61      0.30      0.41      3151
           2       0.76      0.93      0.84      7736

    accuracy                           0.74     10987
   macro avg       0.79      0.41      0.42     10987
weighted avg       0.72      0.74      0.71     10987

Overall Accuracy for 'emotionaldistress': 0.7417



/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [18:07:52] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)


In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, accuracy_score
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

# Load the dataset
data = pd.read_csv('embedded_newalternatedataset.csv')

# Extract features and labels
X = data.drop(columns=['text', 'emotionaldistress', 'provokingviolence', 'individualharrassment']).values  # Features are the TF-IDF columns
y = data[['emotionaldistress', 'provokingviolence', 'individualharrassment']].values  # Multi-label target columns

# Binarize the multi-label targets (if not already binarized)
y_binary = (y > 0).astype(int)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y_binary, test_size=0.2, random_state=42)

# Create a custom Dataset class for PyTorch
class MultilabelDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input': torch.tensor(self.features[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# Create DataLoader for training and validation
train_dataset = MultilabelDataset(X_train, y_train)
val_dataset = MultilabelDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(MLPClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)  # First hidden layer
        self.fc2 = nn.Linear(128, 64)         # Second hidden layer
        self.fc3 = nn.Linear(64, output_dim)  # Output layer
        self.dropout = nn.Dropout(0.3)        # Dropout for regularization

    def forward(self, x):
        x = torch.relu(self.fc1(x))           # Apply ReLU activation
        x = self.dropout(x)                   # Apply dropout
        x = torch.relu(self.fc2(x))           # Apply ReLU activation
        x = self.fc3(x)                       # Output layer
        return torch.sigmoid(x)               # Sigmoid activation for multi-label classification
# Model initialization
input_dim = X.shape[1]  # Number of features (from TF-IDF)
output_dim = y_binary.shape[1]  # Number of target labels (emotionaldistress, provokingviolence, individualharrassment)

model = MLPClassifier(input_dim, output_dim)

# Loss function and optimizer
criterion = nn.BCELoss()  # Binary Cross Entropy Loss for multi-label classification
optimizer = Adam(model.parameters(), lr=0.001)
# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Training function
def train_model(model, train_loader, criterion, optimizer, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_data = batch['input'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_data)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(train_loader):.4f}")

# Train the model
train_model(model, train_loader, criterion, optimizer, epochs=10)
# Evaluation function
def evaluate_model(model, val_loader):
    model.eval()
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_data = batch['input'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_data)
            predictions.append(outputs.cpu().numpy())
            true_labels.append(labels.cpu().numpy())

    return np.vstack(predictions), np.vstack(true_labels)

# Evaluate the model
y_pred, y_true = evaluate_model(model, val_loader)

# Apply threshold to convert outputs to binary predictions
y_pred_binary = (y_pred > 0.5).astype(int)

# Print classification report
print(classification_report(y_true, y_pred_binary, target_names=['Emotional Distress', 'Provoking Violence', 'Individual Harassment']))

# Calculate overall accuracy
overall_accuracy = accuracy_score(y_true, y_pred_binary)
print(f"Overall Accuracy: {overall_accuracy:.4f}")


Epoch 1/10, Loss: 0.7321
Epoch 2/10, Loss: 0.1821
Epoch 3/10, Loss: 0.1821
Epoch 4/10, Loss: 0.1820
Epoch 5/10, Loss: 0.1833
Epoch 6/10, Loss: 0.1821
Epoch 7/10, Loss: 0.1829
Epoch 8/10, Loss: 0.1844
Epoch 9/10, Loss: 0.1821
Epoch 10/10, Loss: 0.1817
                       precision    recall  f1-score   support

   Emotional Distress       0.99      1.00      1.00     10887
   Provoking Violence       0.82      1.00      0.90      9012
Individual Harassment       0.99      1.00      1.00     10906

            micro avg       0.93      1.00      0.97     30805
            macro avg       0.93      1.00      0.96     30805
         weighted avg       0.94      1.00      0.97     30805
          samples avg       0.93      0.99      0.96     30805

Overall Accuracy: 0.8200


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [17]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Define a custom Dataset class
class MultilabelDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input': torch.tensor(self.features[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# Load your dataset
data = pd.read_csv('embedded_newalternatedataset.csv')

# Extract features and labels
X = data.drop(columns=['text', 'emotionaldistress', 'provokingviolence', 'individualharrassment']).values  # TF-IDF features
y = data[['emotionaldistress', 'provokingviolence', 'individualharrassment']].values  # Labels

# Binarize the multi-label targets
y_binary = (y > 0).astype(int)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y_binary, test_size=0.1, random_state=42)

# Create DataLoaders for training and validation
train_dataset = MultilabelDataset(X_train, y_train)
val_dataset = MultilabelDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, input_dim, output_dim, kernel_size=3, hidden_size=128, dropout_rate=0.3):
        super(CNN_BiLSTM_Attention, self).__init__()

        # CNN Layer
        self.cnn = nn.Conv1d(input_dim, hidden_size, kernel_size=kernel_size, padding=kernel_size//2)

        # BiLSTM Layer
        self.bilstm = nn.LSTM(hidden_size, hidden_size, bidirectional=True, batch_first=True)

        # Attention Layer
        self.attention = nn.Linear(hidden_size * 2, 1)

        # Fully connected layers
        self.fc1 = nn.Linear(hidden_size * 2, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_dim)

        # Dropout
        self.dropout = nn.Dropout(dropout_rate)

    def attention_weights(self, lstm_out):
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        return attn_weights

    def forward(self, x):
        # Check if input is already 2D (batch_size, input_dim)
        if len(x.shape) == 2:
            # If input has shape (batch_size, input_dim), reshape it to (batch_size, input_dim, 1) to pass to CNN
            x = x.unsqueeze(1)  # Add a channel dimension

        # CNN Layer
        x = x.transpose(1, 2)  # Change to (batch_size, channels, seq_len) for CNN
        cnn_out = F.relu(self.cnn(x))
        cnn_out = cnn_out.transpose(1, 2)  # (batch_size, seq_len, hidden_size)

        # BiLSTM Layer
        lstm_out, _ = self.bilstm(cnn_out)

        # Attention Mechanism
        attn_weights = self.attention_weights(lstm_out)
        weighted_sum = torch.sum(lstm_out * attn_weights, dim=1)

        # Fully connected layers
        x = F.relu(self.fc1(weighted_sum))
        x = self.dropout(x)
        x = torch.sigmoid(self.fc2(x))  # Sigmoid activation for multi-label classification

        return x

# Model initialization
input_dim = X.shape[1]  # Number of TF-IDF features
output_dim = y_binary.shape[1]  # Number of target labels (3 in this case)

model = CNN_BiLSTM_Attention(input_dim, output_dim)

# Loss function and optimizer
criterion = nn.BCELoss()  # Binary Cross-Entropy loss for multi-label classification
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
def train_model(model, train_loader, criterion, optimizer, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_data = batch['input'].to(device)
            labels = batch['label'].to(device)

            # Forward pass
            outputs = model(input_data)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Backward pass and optimization
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

# Train the model
train_model(model, train_loader, criterion, optimizer, epochs=10)
def evaluate_model(model, val_loader):
    model.eval()
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_data = batch['input'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_data)
            predictions.append(outputs.cpu().numpy())
            true_labels.append(labels.cpu().numpy())

    return np.vstack(predictions), np.vstack(true_labels)

# Evaluate the model
y_pred, y_true = evaluate_model(model, val_loader)

# Apply threshold to convert outputs to binary predictions
y_pred_binary = (y_pred > 0.5).astype(int)

# Print classification report
print(classification_report(y_true, y_pred_binary, target_names=['Emotional Distress', 'Provoking Violence', 'Individual Harassment']))

# Calculate overall accuracy
overall_accuracy = accuracy_score(y_true, y_pred_binary)
print(f"Overall Accuracy: {overall_accuracy:.4f}")


Epoch 1/10, Loss: 0.1889
Epoch 2/10, Loss: 0.1855
Epoch 3/10, Loss: 0.1861
Epoch 4/10, Loss: 0.1852
Epoch 5/10, Loss: 0.1853
Epoch 6/10, Loss: 0.1859
Epoch 7/10, Loss: 0.1850
Epoch 8/10, Loss: 0.1852
Epoch 9/10, Loss: 0.1851
Epoch 10/10, Loss: 0.1848
                       precision    recall  f1-score   support

   Emotional Distress       0.99      1.00      1.00      5444
   Provoking Violence       0.82      1.00      0.90      4505
Individual Harassment       0.99      1.00      1.00      5453

            micro avg       0.93      1.00      0.97     15402
            macro avg       0.93      1.00      0.96     15402
         weighted avg       0.94      1.00      0.97     15402
          samples avg       0.93      0.99      0.96     15402

Overall Accuracy: 0.8196


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
